# Fixed Fractional Position Sizing with Drawdown Cap on SPY
## Strategy Brief -- 3-5 plain-English sentences on signal, prediction, trade logic, results.
This strategy uses fixed fractional position sizing with a drawdown cap to manage risk while trading the SPY ETF. The signal is based on a simple moving average crossover: a buy signal is generated when the short-term moving average crosses above the long-term moving average, and a sell signal is generated when it crosses below. The strategy aims to limit drawdown by capping the maximum allowable loss from the peak. Results are evaluated based on performance metrics such as CAGR, Sharpe ratio, and maximum drawdown.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
Define the trading parameters and constants used throughout the strategy.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SHORT_WINDOW = 50
LONG_WINDOW = 200
FRACTION = 0.02  # 2% of equity per trade
DRAWDOWN_CAP = 0.2  # 20% max drawdown

### PHASE 2 - Data Exploration
Download SPY data from Yahoo Finance, compute moving averages, and plot them overlaid on the price.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Compute moving averages
data['Short_MA'] = data['Close'].rolling(window=SHORT_WINDOW).mean()
data['Long_MA'] = data['Close'].rolling(window=LONG_WINDOW).mean()

# Plot
data[['Close', 'Short_MA', 'Long_MA']].plot(figsize=(14, 7))
plt.title('SPY Price with Moving Averages')
plt.show()

### PHASE 3 - Strategy Engineering
Create signal series based on moving average crossovers and define entry/exit logic.

In [ ]:
# Generate signals
data['Signal'] = 0

data.loc[data['Short_MA'] > data['Long_MA'], 'Signal'] = 1

data.loc[data['Short_MA'] < data['Long_MA'], 'Signal'] = -1

# Define positions based on signals
data['Position'] = data['Signal'].replace(to_replace=0, method='ffill')

### PHASE 4 - Coding & Backtesting
Shift positions, calculate daily returns, and plot the equity curve.

In [ ]:
# Shift positions to avoid lookahead bias
data['Position'] = data['Position'].shift(1)

data['Daily_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Position'] * data['Daily_Return'] * FRACTION

data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()

# Plot equity curve
data['Equity_Curve'].plot(figsize=(14, 7))
plt.title('Equity Curve of Strategy')
plt.show()

### PHASE 5 - Performance Evaluation
Calculate performance metrics and compare against buy-and-hold.

In [ ]:
def calculate_performance_metrics(data):
    cagr = (data['Equity_Curve'].iloc[-1]) ** (1 / ((data.index[-1] - data.index[0]).days / 365.25)) - 1
    annual_volatility = data['Strategy_Return'].std() * np.sqrt(252)
    sharpe_ratio = (data['Strategy_Return'].mean() / data['Strategy_Return'].std()) * np.sqrt(252)
    downside_std = data[data['Strategy_Return'] < 0]['Strategy_Return'].std() * np.sqrt(252)
    sortino_ratio = (data['Strategy_Return'].mean() / downside_std) * np.sqrt(252)
    max_drawdown = (data['Equity_Curve'].cummax() - data['Equity_Curve']).max()
    calmar_ratio = cagr / max_drawdown
    
    return {
        'CAGR': cagr,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Calmar Ratio': calmar_ratio,
        'Max Drawdown': max_drawdown
    }

performance_metrics = calculate_performance_metrics(data)

# Buy-and-hold comparison
data['Buy_Hold'] = (1 + data['Daily_Return']).cumprod()
buy_hold_cagr = (data['Buy_Hold'].iloc[-1]) ** (1 / ((data.index[-1] - data.index[0]).days / 365.25)) - 1

comparison_table = pd.DataFrame({
    'Strategy': performance_metrics,
    'Buy & Hold': {
        'CAGR': buy_hold_cagr,
        'Sharpe Ratio': (data['Daily_Return'].mean() / data['Daily_Return'].std()) * np.sqrt(252),
        'Sortino Ratio': (data['Daily_Return'].mean() / downside_std) * np.sqrt(252),
        'Calmar Ratio': buy_hold_cagr / max_drawdown,
        'Max Drawdown': max_drawdown
    }
})

comparison_table

### PHASE 6 - Deploy & Monitor
Create a function to download the last 60 days of data, compute today's signal, and print the position.

In [ ]:
def get_latest_signal():
    latest_data = yf.download('SPY', period='60d')
    latest_data['Short_MA'] = latest_data['Close'].rolling(window=SHORT_WINDOW).mean()
    latest_data['Long_MA'] = latest_data['Close'].rolling(window=LONG_WINDOW).mean()
    
    if latest_data['Short_MA'].iloc[-1] > latest_data['Long_MA'].iloc[-1]:
        print('Current Position: Long')
    elif latest_data['Short_MA'].iloc[-1] < latest_data['Long_MA'].iloc[-1]:
        print('Current Position: Short')
    else:
        print('Current Position: Neutral')

get_latest_signal()